In [ ]:
# Self-contained resume-capable SGCN training + checkpoints + learning curves
# ------------------------------------------------------------------------
# This cell can run independently (no dependency on previous notebook cells).
# Preferred mode: deep node attributes + raw-feature-driven dynamic edge weights.

from pathlib import Path
from functools import lru_cache
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import matplotlib.pyplot as plt
from time import perf_counter


# -------------------------
# Reproducibility + device
# -------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")


# -------------------------
# Data paths
# -------------------------
ROOT = Path("FGVD_Graph_Handover")
META_PATH = ROOT / "metadata.csv"
ADJ_PATH = ROOT / "master_grid_skeleton.npz"
RAW_FEAT_ROOT = ROOT / "raw_features"
DEEP_FEAT_ROOT = ROOT / "deep_features"
DEEP_NODE_FEATURE_SUBDIR = "multilevel"

if not META_PATH.exists():
    raise FileNotFoundError(f"Missing metadata: {META_PATH}")
if not ADJ_PATH.exists():
    raise FileNotFoundError(f"Missing adjacency: {ADJ_PATH}")
if not RAW_FEAT_ROOT.exists():
    raise FileNotFoundError(f"Missing raw feature root: {RAW_FEAT_ROOT}")
if not DEEP_FEAT_ROOT.exists():
    raise FileNotFoundError(f"Missing deep feature root: {DEEP_FEAT_ROOT}")

metadata = pd.read_csv(META_PATH)
for c in ["vehicle_id", "split", "L1", "L2", "L3"]:
    metadata[c] = metadata[c].astype(str).str.strip()

print("Rows:", len(metadata))
print("Split counts:", metadata["split"].value_counts().to_dict())

# master adjacency (same sparse neighborhood topology for all samples)
adj_sparse = sp.load_npz(ADJ_PATH).tocoo()
edge_index = torch.tensor(np.vstack([adj_sparse.row, adj_sparse.col]), dtype=torch.long)
print("Adj shape:", adj_sparse.shape, "| edges:", edge_index.shape[1])


# -------------------------
# Label levels and cases
# -------------------------
RAW_EDGE_FEATURES = {
    "L1": ("rgb", "gabor", "sobel"),
    "L2": ("rgb", "gabor"),
    "L3": ("rgb", "gabor"),
}

TW_SET = {"motorcycle", "scooter"}
THW_SET = {"autorickshaw", "auto", "threewheeler", "three_wheeler"}


def norm_l1(x: str) -> str:
    return str(x).strip().lower().replace("-", "").replace(" ", "")


def build_level_labels(df: pd.DataFrame, level: str) -> np.ndarray:
    if level == "L1":
        return df["L1"].to_numpy(dtype=object)
    if level == "L2":
        return (df["L1"] + "::" + df["L2"]).to_numpy(dtype=object)
    if level == "L3":
        return (df["L1"] + "::" + df["L2"] + "::" + df["L3"]).to_numpy(dtype=object)
    raise ValueError("level must be L1/L2/L3")


def apply_case(df: pd.DataFrame, base_labels: np.ndarray, level: str, case_name: str):
    l1_norm = df["L1"].map(norm_l1).to_numpy()
    is_tw = np.isin(l1_norm, list(TW_SET))
    is_thw = np.isin(l1_norm, list(THW_SET))
    is_car = l1_norm == "car"

    if case_name == "all":
        keep = np.ones(len(df), dtype=bool)
        return base_labels, keep

    if case_name == "tw_vs_car":
        keep = is_tw | is_car
        return base_labels[keep], keep

    if case_name == "tw_vs_all":
        if level == "L1":
            y = np.where(is_tw, "two_wheeler", "other")
        else:
            y = np.where(is_tw, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    if case_name == "thw_vs_all":
        if level == "L1":
            y = np.where(is_thw, "three_wheeler", "other")
        else:
            y = np.where(is_thw, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    if case_name == "tw_thw_vs_all":
        group = is_tw | is_thw
        if level == "L1":
            y = np.where(group, "two_or_three_wheeler", "other")
        else:
            y = np.where(group, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    raise ValueError(f"Unknown case_name: {case_name}")


# -------------------------
# Feature loading + checks
# -------------------------
@lru_cache(maxsize=20000)
def _raw_feature_path(vehicle_id: str, split: str, feat: str) -> Path:
    return RAW_FEAT_ROOT / feat / split / f"{vehicle_id}.npy"


@lru_cache(maxsize=20000)
def _deep_feature_path(vehicle_id: str, split: str, deep_subdir: str) -> Path:
    return DEEP_FEAT_ROOT / deep_subdir / split / f"{vehicle_id}.npy"


@lru_cache(maxsize=40000)
def _has_raw_feature_file(vehicle_id: str, split: str, feat: str) -> bool:
    return _raw_feature_path(vehicle_id, split, feat).exists()


@lru_cache(maxsize=20000)
def _has_deep_feature_file(vehicle_id: str, split: str, deep_subdir: str) -> bool:
    return _deep_feature_path(vehicle_id, split, deep_subdir).exists()


def _load_raw_feature(vehicle_id: str, split: str, feat: str):
    # Avoid caching full arrays across epochs to prevent RAM growth/hangs.
    p = _raw_feature_path(vehicle_id, split, feat)
    if not p.exists():
        raise FileNotFoundError(f"Missing raw feature file: {p}")
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)


def _load_deep_feature(vehicle_id: str, split: str, deep_subdir: str):
    # Avoid caching full arrays across epochs to prevent RAM growth/hangs.
    p = _deep_feature_path(vehicle_id, split, deep_subdir)
    if not p.exists():
        raise FileNotFoundError(f"Missing deep feature file: {p}")
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)


def load_raw_stacked_features(vehicle_id: str, split: str, feature_names):
    parts = [_load_raw_feature(vehicle_id, split, feat) for feat in feature_names]
    return np.concatenate(parts, axis=1)


def load_node_features(vehicle_id: str, split: str, deep_subdir: str, raw_feature_names, node_feature_source: str):
    if node_feature_source == "deep":
        return _load_deep_feature(vehicle_id, split, deep_subdir)
    if node_feature_source == "raw":
        return load_raw_stacked_features(vehicle_id, split, raw_feature_names)
    raise ValueError(f"Unknown node_feature_source: {node_feature_source}")


def _raw_available(vehicle_id: str, split: str, raw_feature_names) -> bool:
    return all(_has_raw_feature_file(vehicle_id, split, feat) for feat in raw_feature_names)


def _deep_available(vehicle_id: str, split: str, deep_subdir: str) -> bool:
    return _has_deep_feature_file(vehicle_id, split, deep_subdir)


def _build_availability_masks(df: pd.DataFrame, raw_feature_names, deep_subdir: str):
    raw_mask = []
    deep_and_raw_mask = []
    total = len(df)
    started = perf_counter()
    for _, row in df.iterrows():
        vid = str(row["vehicle_id"])
        sp_name = str(row["split"])
        has_raw = _raw_available(vid, sp_name, raw_feature_names)
        has_deep = _deep_available(vid, sp_name, deep_subdir)
        raw_mask.append(has_raw)
        deep_and_raw_mask.append(has_raw and has_deep)
        if total and len(raw_mask) % 500 == 0:
            elapsed = perf_counter() - started
            print(f"  scanned {len(raw_mask)}/{total} rows for feature availability ({elapsed:.1f}s)", flush=True)
    return np.asarray(raw_mask, dtype=bool), np.asarray(deep_and_raw_mask, dtype=bool)


def _apply_mask(df: pd.DataFrame, y_raw: np.ndarray, mask: np.ndarray):
    return df.loc[mask].reset_index(drop=True), y_raw[mask]


# -------------------------
# Dataset / model / epoch
# -------------------------
class FGVDGraphDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        y_encoded: np.ndarray,
        raw_edge_feature_names,
        deep_feature_subdir: str,
        node_feature_source: str,
    ):
        self.df = df.reset_index(drop=True)
        self.y = y_encoded
        self.raw_edge_feature_names = tuple(raw_edge_feature_names)
        self.deep_feature_subdir = str(deep_feature_subdir)
        self.node_feature_source = str(node_feature_source)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vehicle_id = str(row["vehicle_id"])
        split = str(row["split"])

        # Node attributes: deep features by default; raw fallback only if configured.
        x_np = load_node_features(
            vehicle_id,
            split,
            self.deep_feature_subdir,
            self.raw_edge_feature_names,
            self.node_feature_source,
        )
        x = torch.tensor(x_np, dtype=torch.float32)

        # Raw features are also returned so edge weights can be computed on GPU.
        raw_np = load_raw_stacked_features(vehicle_id, split, self.raw_edge_feature_names)
        raw_x = torch.tensor(raw_np, dtype=torch.float32)

        y = torch.tensor(int(self.y[idx]), dtype=torch.long)
        return Data(x=x, raw_x=raw_x, edge_index=edge_index, y=y)


class SGCNLayer(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.5):
        super().__init__()
        self.conv = GCNConv(in_channels, out_channels, add_self_loops=True, normalize=True)
        self.bn = nn.BatchNorm1d(out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr=None):
        x = self.conv(x, edge_index, edge_weight=edge_attr)
        x = self.bn(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class SGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.5,
        edge_sigma: float = 0.5,
    ):
        super().__init__()
        self.edge_sigma = float(edge_sigma)
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(
                SGCNLayer(
                    in_channels if i == 0 else hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                )
            )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def _compute_dynamic_edge_attr(self, raw_x, edge_index):
        u, v = edge_index[0], edge_index[1]
        dist = torch.norm(raw_x[u] - raw_x[v], dim=1)
        sigma = max(self.edge_sigma, 1e-6)
        return torch.exp(-(dist ** 2) / (2.0 * (sigma ** 2)))

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = getattr(data, "batch", None)
        raw_x = getattr(data, "raw_x", None)

        if raw_x is None:
            raise RuntimeError("Expected 'raw_x' in Data batch for dynamic edge computation.")

        # Dynamic edge weights are computed on GPU from raw features.
        edge_attr = self._compute_dynamic_edge_attr(raw_x, edge_index)

        for layer in self.layers:
            x = layer(x, edge_index, edge_attr)

        x = global_mean_pool(x, batch)
        return self.classifier(x)


def run_epoch(model, loader, criterion, optimizer=None, use_amp: bool = True):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    preds_all, targets_all = [], []

    amp_enabled = use_amp and device.type == "cuda"

    for batch in loader:
        batch = batch.to(device, non_blocking=(device.type == "cuda"))
        targets = batch.y.view(-1)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(batch)
                loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        preds = logits.argmax(dim=1)
        preds_all.extend(preds.detach().cpu().numpy().tolist())
        targets_all.extend(targets.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")
    return avg_loss, acc, macro_f1, np.asarray(preds_all), np.asarray(targets_all)


# -------------------------
# Experiment preparation
# -------------------------
def _tag(level: str, case_name: str) -> str:
    return f"{level}_{case_name}".replace("/", "_").replace(" ", "_")


def prepare_experiment(
    level: str,
    case_name: str,
    batch_size: int = 32,
    num_workers: int = 0,
    deep_feature_subdir: str = DEEP_NODE_FEATURE_SUBDIR,
    allow_raw_node_fallback: bool = True,
):
    train_df = metadata[metadata["split"] == "train"].copy()
    val_df = metadata[metadata["split"] == "val"].copy()
    test_df = metadata[metadata["split"] == "test"].copy()

    y_train_base = build_level_labels(train_df, level)
    y_val_base = build_level_labels(val_df, level)
    y_test_base = build_level_labels(test_df, level)

    y_train_raw, keep_train = apply_case(train_df, y_train_base, level, case_name)
    y_val_raw, keep_val = apply_case(val_df, y_val_base, level, case_name)
    y_test_raw, keep_test = apply_case(test_df, y_test_base, level, case_name)

    train_df = train_df.loc[keep_train].reset_index(drop=True)
    val_df = val_df.loc[keep_val].reset_index(drop=True)
    test_df = test_df.loc[keep_test].reset_index(drop=True)

    raw_edge_feature_names = RAW_EDGE_FEATURES[level]

    train_raw_mask, train_deep_raw_mask = _build_availability_masks(train_df, raw_edge_feature_names, deep_feature_subdir)
    val_raw_mask, val_deep_raw_mask = _build_availability_masks(val_df, raw_edge_feature_names, deep_feature_subdir)
    test_raw_mask, test_deep_raw_mask = _build_availability_masks(test_df, raw_edge_feature_names, deep_feature_subdir)

    deep_ready = train_deep_raw_mask.any() and val_deep_raw_mask.any() and test_deep_raw_mask.any()

    if deep_ready:
        node_feature_source = "deep"
        train_df, y_train_raw = _apply_mask(train_df, y_train_raw, train_deep_raw_mask)
        val_df, y_val_raw = _apply_mask(val_df, y_val_raw, val_deep_raw_mask)
        test_df, y_test_raw = _apply_mask(test_df, y_test_raw, test_deep_raw_mask)
    else:
        if not allow_raw_node_fallback:
            raise RuntimeError(
                "Deep feature files are insufficient across train/val/test after filtering. "
                "Enable allow_raw_node_fallback or generate missing deep features."
            )
        node_feature_source = "raw"
        train_df, y_train_raw = _apply_mask(train_df, y_train_raw, train_raw_mask)
        val_df, y_val_raw = _apply_mask(val_df, y_val_raw, val_raw_mask)
        test_df, y_test_raw = _apply_mask(test_df, y_test_raw, test_raw_mask)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise RuntimeError(
            f"No usable samples after feature-file filtering. "
            f"Counts train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}"
        )

    le = LabelEncoder()
    le.fit(np.concatenate([y_train_raw, y_val_raw, y_test_raw]))

    y_train = le.transform(y_train_raw)
    y_val = le.transform(y_val_raw)
    y_test = le.transform(y_test_raw)

    train_dataset = FGVDGraphDataset(
        train_df,
        y_train,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )
    val_dataset = FGVDGraphDataset(
        val_df,
        y_val,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )
    test_dataset = FGVDGraphDataset(
        test_df,
        y_test,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )

    loader_kwargs = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": device.type == "cuda",
    }
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 4

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

    sample_source_df = train_df if len(train_df) > 0 else (val_df if len(val_df) > 0 else test_df)
    sample_vehicle_id = str(sample_source_df.iloc[0]["vehicle_id"])
    sample_split = str(sample_source_df.iloc[0]["split"])

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "label_encoder": le,
        "train_count": len(train_df),
        "val_count": len(val_df),
        "test_count": len(test_df),
        "num_classes": len(le.classes_),
        "sample_vehicle_id": sample_vehicle_id,
        "sample_split": sample_split,
        "deep_feature_subdir": deep_feature_subdir,
        "raw_edge_feature_names": raw_edge_feature_names,
        "node_feature_source": node_feature_source,
    }


# -------------------------
# Plotting
# -------------------------
def plot_learning_curves(history: dict, title: str, out_path: Path):
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(epochs, history["train_loss"], label="train_loss", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], label="val_loss", linewidth=2)
    axes[0].set_title("Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="train_acc", linewidth=2)
    axes[1].plot(epochs, history["val_acc"], label="val_acc", linewidth=2)
    axes[1].set_title("Accuracy Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()


# -------------------------
# Main fit function
# -------------------------
def fit_with_resume(
    level: str = "L1",
    case_name: str = "all",
    additional_epochs: int = 20,
    target_total_epochs: int = None,
    resume: bool = True,
    clear_old_checkpoints: bool = False,
    batch_size: int = 32,
    hidden_dim: int = 64,
    num_layers: int = 2,
    dropout: float = 0.5,
    lr: float = 1e-3,
    num_workers: int = 0,
    print_every: int = 5,
    deep_feature_subdir: str = DEEP_NODE_FEATURE_SUBDIR,
    edge_sigma: float = 0.5,
    allow_raw_node_fallback: bool = True,
    use_amp: bool = True,
    run_name: str = "fresh_start",
    save_curve: bool = False,
):
    run_tag = str(run_name).strip().replace(" ", "_") or "fresh_start"
    ckpt_root = Path("checkpoints/sgcn_df") / run_tag
    ckpt_root.mkdir(parents=True, exist_ok=True)

    curve_png = None
    if save_curve:
        plot_root = Path("plots/sgcn_df") / run_tag
        plot_root.mkdir(parents=True, exist_ok=True)

    tag = _tag(level, case_name)
    last_ckpt = ckpt_root / f"{tag}_last.pt"
    best_ckpt = ckpt_root / f"{tag}_best.pt"
    if save_curve:
        curve_png = plot_root / f"{tag}_curves.png"

    if clear_old_checkpoints:
        if last_ckpt.exists():
            last_ckpt.unlink()
        if best_ckpt.exists():
            best_ckpt.unlink()

    exp = prepare_experiment(
        level,
        case_name,
        batch_size=batch_size,
        num_workers=num_workers,
        deep_feature_subdir=deep_feature_subdir,
        allow_raw_node_fallback=allow_raw_node_fallback,
    )
    print("Feature availability scan complete.", flush=True)
    print(f"\nRunning {level} | {case_name}")
    print(f"Samples: train={exp['train_count']}, val={exp['val_count']}, test={exp['test_count']}")
    print(f"Classes: {exp['num_classes']}")
    print(f"Node features source: {exp['node_feature_source']}")
    print(f"Node deep subdir: {exp['deep_feature_subdir']}")
    print(f"Edge features (raw): {exp['raw_edge_feature_names']}")
    print(f"Gaussian sigma: {edge_sigma}")
    print(f"AMP enabled: {use_amp and device.type == 'cuda'}")

    sample_x = load_node_features(
        exp["sample_vehicle_id"],
        exp["sample_split"],
        exp["deep_feature_subdir"],
        exp["raw_edge_feature_names"],
        exp["node_feature_source"],
    )
    in_channels = int(sample_x.shape[1])

    model = SGCN(
        in_channels=in_channels,
        num_classes=exp["num_classes"],
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
        edge_sigma=edge_sigma,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_f1": [],
        "val_f1": [],
    }

    start_epoch = 1
    best_val_acc = -1.0
    best_epoch = 0
    best_model_state = None

    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        history = ckpt.get("history", history)
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val_acc = float(ckpt.get("best_val_acc", -1.0))
        best_epoch = int(ckpt.get("best_epoch", 0))
        best_model_state = ckpt.get("best_model_state_dict", None)

        saved_classes = ckpt.get("label_classes", [])
        current_classes = exp["label_encoder"].classes_.tolist()
        if saved_classes and saved_classes != current_classes:
            raise RuntimeError("Label encoding changed since checkpoint creation. Cannot safely resume.")

        print(f"Resumed from checkpoint: {last_ckpt}")
        print(f"Restarting at epoch {start_epoch}")

    if target_total_epochs is not None:
        end_epoch = int(target_total_epochs)
    else:
        end_epoch = start_epoch + int(additional_epochs) - 1

    if start_epoch > end_epoch:
        print("No new epochs requested. Evaluating current best model.")
    else:
        print(f"Starting training from epoch {start_epoch} to {end_epoch}.", flush=True)
        for epoch in range(start_epoch, end_epoch + 1):
            tr_loss, tr_acc, tr_f1, _, _ = run_epoch(model, exp["train_loader"], criterion, optimizer, use_amp=use_amp)
            va_loss, va_acc, va_f1, _, _ = run_epoch(model, exp["val_loader"], criterion, optimizer=None, use_amp=use_amp)

            history["epoch"].append(epoch)
            history["train_loss"].append(float(tr_loss))
            history["val_loss"].append(float(va_loss))
            history["train_acc"].append(float(tr_acc))
            history["val_acc"].append(float(va_acc))
            history["train_f1"].append(float(tr_f1))
            history["val_f1"].append(float(va_f1))

            if va_acc > best_val_acc:
                best_val_acc = float(va_acc)
                best_epoch = int(epoch)
                best_model_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

                torch.save(
                    {
                        "epoch": epoch,
                        "best_epoch": best_epoch,
                        "best_val_acc": best_val_acc,
                        "best_model_state_dict": best_model_state,
                        "history": history,
                        "label_classes": exp["label_encoder"].classes_.tolist(),
                    },
                    best_ckpt,
                )

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_model_state_dict": best_model_state,
                    "history": history,
                    "label_classes": exp["label_encoder"].classes_.tolist(),
                },
                last_ckpt,
            )

            if device.type == "cuda":
                # Reduces long-run memory fragmentation between epochs.
                torch.cuda.empty_cache()

            if epoch == start_epoch or epoch % print_every == 0 or epoch == end_epoch:
                print(
                    f"Epoch {epoch:03d} | "
                    f"train_acc={tr_acc:.4f} val_acc={va_acc:.4f} | "
                    f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f}"
                )

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    te_loss, te_acc, te_f1, te_pred, te_true = run_epoch(model, exp["test_loader"], criterion, optimizer=None, use_amp=use_amp)

    pred_labels = exp["label_encoder"].inverse_transform(te_pred.astype(int))
    true_labels = exp["label_encoder"].inverse_transform(te_true.astype(int))

    print("\nFinal test metrics (best validation-accuracy checkpoint):")
    print(f"test_acc={te_acc:.4f} | test_f1={te_f1:.4f} | test_loss={te_loss:.4f}")
    print(classification_report(true_labels, pred_labels, digits=4))

    if save_curve and curve_png is not None:
        title = f"SGCN Learning Curves | {level} | {case_name}"
        plot_learning_curves(history, title=title, out_path=curve_png)

    if history["train_acc"]:
        gap = history["train_acc"][-1] - history["val_acc"][-1]
        if history["train_acc"][-1] < 0.70 and history["val_acc"][-1] < 0.70:
            fit_note = "Possible underfitting (both train/val accuracy are low)."
        elif gap > 0.08:
            fit_note = "Possible overfitting (train-val accuracy gap is high)."
        else:
            fit_note = "No strong over/underfitting signal from final epoch gap."
        print(f"Fit check: {fit_note} gap={gap:+.4f}")

    print(f"Saved: {last_ckpt}")
    print(f"Saved: {best_ckpt}")
    print(f"Checkpoint folder: {ckpt_root.resolve()}")
    if save_curve and curve_png is not None:
        print(f"Saved: {curve_png}")
        print(f"Plot folder: {plot_root.resolve()}")
    print(
        f"Checkpoint files present -> last: {last_ckpt.exists()} | best: {best_ckpt.exists()}"
    )

    return {
        "level": level,
        "case": case_name,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "test_acc": float(te_acc),
        "test_f1": float(te_f1),
        "checkpoint_last": str(last_ckpt),
        "checkpoint_best": str(best_ckpt),
        "edge_sigma": float(edge_sigma),
        "node_feature_subdir": deep_feature_subdir,
        "node_feature_source": exp["node_feature_source"],
        "amp": bool(use_amp and device.type == "cuda"),
    }


# -------------------------
# USER CONTROLS
# -------------------------
LEVEL_TO_RUN = "L1"
CASE_TO_RUN = "all"            # all, tw_vs_all, thw_vs_all, tw_thw_vs_all, tw_vs_car
RUN_NAME = "L1_all"
RESUME = False                  # force start from epoch 1
CLEAR_OLD_CHECKPOINTS = True    # clear old files inside this run folder
ADDITIONAL_EPOCHS = 100         # used when TARGET_TOTAL_EPOCHS is None
TARGET_TOTAL_EPOCHS = 300       # exact total epochs target
EDGE_SIGMA = 0.5                # Gaussian kernel sigma for dynamic edge weights
ALLOW_RAW_NODE_FALLBACK = True  # if deep files are insufficient, use raw as node attributes
NUM_WORKERS = 0                 # Set to 0 to prevent RAM exhaustion / system hangs
USE_AMP = True                  # mixed precision on CUDA for faster training
SAVE_CURVE = False              # False => only keep *_best.pt and *_last.pt

result = fit_with_resume(
    level=LEVEL_TO_RUN,
    case_name=CASE_TO_RUN,
    additional_epochs=ADDITIONAL_EPOCHS,
    target_total_epochs=TARGET_TOTAL_EPOCHS,
    resume=RESUME,
    clear_old_checkpoints=CLEAR_OLD_CHECKPOINTS,
    batch_size=32,
    hidden_dim=64,
    num_layers=2,
    dropout=0.5,
    lr=1e-3,
    num_workers=NUM_WORKERS,
    print_every=5,
    deep_feature_subdir=DEEP_NODE_FEATURE_SUBDIR,
    edge_sigma=EDGE_SIGMA,
    allow_raw_node_fallback=ALLOW_RAW_NODE_FALLBACK,
    use_amp=USE_AMP,
    run_name=RUN_NAME,
    save_curve=SAVE_CURVE,
)

print("\nRun result:")
print(result)

/home/cse/Desktop/btech_project/venv_fgvd/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /home/cse/Desktop/btech_project/venv_fgvd/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/home/cse/Desktop/btech_project/venv_fgvd/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /home/cse/Desktop/btech_project/venv_fgvd/lib/python3.12/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/home/cse/Desktop/btech_project/venv_fgvd/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: Could not load this library: /home/cse/Desktop/btech_project/venv_fgvd/li

Using device: cuda


FileNotFoundError: Missing adjacency: FGVD_Graph_Handover/master_grid_adj.npz

In [ ]:
! pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.5.0+cu118.html

Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 356.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 4.4 MB/s eta 0:00:0000:010:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 1.9 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.0 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 582.8 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 125.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 40.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 653.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# Self-contained resume-capable SGCN training + checkpoints + learning curves
# ------------------------------------------------------------------------
# This cell can run independently (no dependency on previous notebook cells).
# Preferred mode: deep node attributes + raw-feature-driven dynamic edge weights.

from pathlib import Path
from functools import lru_cache
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import matplotlib.pyplot as plt


# -------------------------
# Reproducibility + device
# -------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")


# -------------------------
# Data paths
# -------------------------
ROOT = Path("FGVD_Graph_Handover")
META_PATH = ROOT / "metadata.csv"
ADJ_PATH = ROOT / "master_grid_skeleton.npz"
RAW_FEAT_ROOT = ROOT / "raw_features"
DEEP_FEAT_ROOT = ROOT / "deep_features"
DEEP_NODE_FEATURE_SUBDIR = "multilevel"

if not META_PATH.exists():
    raise FileNotFoundError(f"Missing metadata: {META_PATH}")
if not ADJ_PATH.exists():
    raise FileNotFoundError(f"Missing adjacency: {ADJ_PATH}")
if not RAW_FEAT_ROOT.exists():
    raise FileNotFoundError(f"Missing raw feature root: {RAW_FEAT_ROOT}")
if not DEEP_FEAT_ROOT.exists():
    raise FileNotFoundError(f"Missing deep feature root: {DEEP_FEAT_ROOT}")

metadata = pd.read_csv(META_PATH)
for c in ["vehicle_id", "split", "L1", "L2", "L3"]:
    metadata[c] = metadata[c].astype(str).str.strip()

print("Rows:", len(metadata))
print("Split counts:", metadata["split"].value_counts().to_dict())

# master adjacency (same sparse neighborhood topology for all samples)
adj_sparse = sp.load_npz(ADJ_PATH).tocoo()
edge_index = torch.tensor(np.vstack([adj_sparse.row, adj_sparse.col]), dtype=torch.long)
print("Adj shape:", adj_sparse.shape, "| edges:", edge_index.shape[1])


# -------------------------
# Label levels and cases
# -------------------------
RAW_EDGE_FEATURES = {
    "L1": ("rgb", "gabor", "sobel"),
    "L2": ("rgb", "gabor"),
    "L3": ("rgb", "gabor"),
}

TW_SET = {"motorcycle", "scooter"}
THW_SET = {"autorickshaw", "auto", "threewheeler", "three_wheeler"}


def norm_l1(x: str) -> str:
    return str(x).strip().lower().replace("-", "").replace(" ", "")


def build_level_labels(df: pd.DataFrame, level: str) -> np.ndarray:
    if level == "L1":
        return df["L1"].to_numpy(dtype=object)
    if level == "L2":
        return (df["L1"] + "::" + df["L2"]).to_numpy(dtype=object)
    if level == "L3":
        return (df["L1"] + "::" + df["L2"] + "::" + df["L3"]).to_numpy(dtype=object)
    raise ValueError("level must be L1/L2/L3")


def apply_case(df: pd.DataFrame, base_labels: np.ndarray, level: str, case_name: str):
    l1_norm = df["L1"].map(norm_l1).to_numpy()
    is_tw = np.isin(l1_norm, list(TW_SET))
    is_thw = np.isin(l1_norm, list(THW_SET))
    is_car = l1_norm == "car"

    if case_name == "all":
        keep = np.ones(len(df), dtype=bool)
        return base_labels, keep

    if case_name == "tw_vs_car":
        keep = is_tw | is_car
        return base_labels[keep], keep

    if case_name == "tw_vs_all":
        if level == "L1":
            y = np.where(is_tw, "two_wheeler", "other")
        else:
            y = np.where(is_tw, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    if case_name == "thw_vs_all":
        if level == "L1":
            y = np.where(is_thw, "three_wheeler", "other")
        else:
            y = np.where(is_thw, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    if case_name == "tw_thw_vs_all":
        group = is_tw | is_thw
        if level == "L1":
            y = np.where(group, "two_or_three_wheeler", "other")
        else:
            y = np.where(group, base_labels, "other")
        keep = np.ones(len(df), dtype=bool)
        return y, keep

    raise ValueError(f"Unknown case_name: {case_name}")


# -------------------------
# Feature loading + checks
# -------------------------
@lru_cache(maxsize=20000)
def _raw_feature_path(vehicle_id: str, split: str, feat: str) -> Path:
    return RAW_FEAT_ROOT / feat / split / f"{vehicle_id}.npy"


@lru_cache(maxsize=20000)
def _deep_feature_path(vehicle_id: str, split: str, deep_subdir: str) -> Path:
    return DEEP_FEAT_ROOT / deep_subdir / split / f"{vehicle_id}.npy"


@lru_cache(maxsize=40000)
def _has_raw_feature_file(vehicle_id: str, split: str, feat: str) -> bool:
    return _raw_feature_path(vehicle_id, split, feat).exists()


@lru_cache(maxsize=20000)
def _has_deep_feature_file(vehicle_id: str, split: str, deep_subdir: str) -> bool:
    return _deep_feature_path(vehicle_id, split, deep_subdir).exists()


def _load_raw_feature(vehicle_id: str, split: str, feat: str):
    # Avoid caching full arrays across epochs to prevent RAM growth/hangs.
    p = _raw_feature_path(vehicle_id, split, feat)
    if not p.exists():
        raise FileNotFoundError(f"Missing raw feature file: {p}")
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)


def _load_deep_feature(vehicle_id: str, split: str, deep_subdir: str):
    # Avoid caching full arrays across epochs to prevent RAM growth/hangs.
    p = _deep_feature_path(vehicle_id, split, deep_subdir)
    if not p.exists():
        raise FileNotFoundError(f"Missing deep feature file: {p}")
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)


def load_raw_stacked_features(vehicle_id: str, split: str, feature_names):
    parts = [_load_raw_feature(vehicle_id, split, feat) for feat in feature_names]
    return np.concatenate(parts, axis=1)


def load_node_features(vehicle_id: str, split: str, deep_subdir: str, raw_feature_names, node_feature_source: str):
    if node_feature_source == "deep":
        return _load_deep_feature(vehicle_id, split, deep_subdir)
    if node_feature_source == "raw":
        return load_raw_stacked_features(vehicle_id, split, raw_feature_names)
    raise ValueError(f"Unknown node_feature_source: {node_feature_source}")


def _raw_available(vehicle_id: str, split: str, raw_feature_names) -> bool:
    return all(_has_raw_feature_file(vehicle_id, split, feat) for feat in raw_feature_names)


def _deep_available(vehicle_id: str, split: str, deep_subdir: str) -> bool:
    return _has_deep_feature_file(vehicle_id, split, deep_subdir)


def _build_availability_masks(df: pd.DataFrame, raw_feature_names, deep_subdir: str):
    raw_mask = []
    deep_and_raw_mask = []
    for _, row in df.iterrows():
        vid = str(row["vehicle_id"])
        sp_name = str(row["split"])
        has_raw = _raw_available(vid, sp_name, raw_feature_names)
        has_deep = _deep_available(vid, sp_name, deep_subdir)
        raw_mask.append(has_raw)
        deep_and_raw_mask.append(has_raw and has_deep)
    return np.asarray(raw_mask, dtype=bool), np.asarray(deep_and_raw_mask, dtype=bool)


def _apply_mask(df: pd.DataFrame, y_raw: np.ndarray, mask: np.ndarray):
    return df.loc[mask].reset_index(drop=True), y_raw[mask]


# -------------------------
# Dataset / model / epoch
# -------------------------
class FGVDGraphDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        y_encoded: np.ndarray,
        raw_edge_feature_names,
        deep_feature_subdir: str,
        node_feature_source: str,
    ):
        self.df = df.reset_index(drop=True)
        self.y = y_encoded
        self.raw_edge_feature_names = tuple(raw_edge_feature_names)
        self.deep_feature_subdir = str(deep_feature_subdir)
        self.node_feature_source = str(node_feature_source)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vehicle_id = str(row["vehicle_id"])
        split = str(row["split"])

        # Node attributes: deep features by default; raw fallback only if configured.
        x_np = load_node_features(
            vehicle_id,
            split,
            self.deep_feature_subdir,
            self.raw_edge_feature_names,
            self.node_feature_source,
        )
        x = torch.tensor(x_np, dtype=torch.float32)

        # Raw features are also returned so edge weights can be computed on GPU.
        raw_np = load_raw_stacked_features(vehicle_id, split, self.raw_edge_feature_names)
        raw_x = torch.tensor(raw_np, dtype=torch.float32)

        y = torch.tensor(int(self.y[idx]), dtype=torch.long)
        return Data(x=x, raw_x=raw_x, edge_index=edge_index, y=y)


class SGCNLayer(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.5):
        super().__init__()
        self.conv = GCNConv(in_channels, out_channels, add_self_loops=True, normalize=True)
        self.bn = nn.BatchNorm1d(out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr=None):
        x = self.conv(x, edge_index, edge_weight=edge_attr)
        x = self.bn(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class SGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.5,
        edge_sigma: float = 0.5,
    ):
        super().__init__()
        self.edge_sigma = float(edge_sigma)
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(
                SGCNLayer(
                    in_channels if i == 0 else hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                )
            )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def _compute_dynamic_edge_attr(self, raw_x, edge_index):
        u, v = edge_index[0], edge_index[1]
        dist = torch.norm(raw_x[u] - raw_x[v], dim=1)
        sigma = max(self.edge_sigma, 1e-6)
        return torch.exp(-(dist ** 2) / (2.0 * (sigma ** 2)))

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = getattr(data, "batch", None)
        raw_x = getattr(data, "raw_x", None)

        if raw_x is None:
            raise RuntimeError("Expected 'raw_x' in Data batch for dynamic edge computation.")

        # Dynamic edge weights are computed on GPU from raw features.
        edge_attr = self._compute_dynamic_edge_attr(raw_x, edge_index)

        for layer in self.layers:
            x = layer(x, edge_index, edge_attr)

        x = global_mean_pool(x, batch)
        return self.classifier(x)


def run_epoch(model, loader, criterion, optimizer=None, use_amp: bool = True):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    preds_all, targets_all = [], []

    amp_enabled = use_amp and device.type == "cuda"

    for batch in loader:
        batch = batch.to(device, non_blocking=(device.type == "cuda"))
        targets = batch.y.view(-1)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(batch)
                loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        preds = logits.argmax(dim=1)
        preds_all.extend(preds.detach().cpu().numpy().tolist())
        targets_all.extend(targets.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")
    return avg_loss, acc, macro_f1, np.asarray(preds_all), np.asarray(targets_all)


# -------------------------
# Experiment preparation
# -------------------------
def _tag(level: str, case_name: str) -> str:
    return f"{level}_{case_name}".replace("/", "_").replace(" ", "_")


def prepare_experiment(
    level: str,
    case_name: str,
    batch_size: int = 32,
    num_workers: int = 0,
    deep_feature_subdir: str = DEEP_NODE_FEATURE_SUBDIR,
    allow_raw_node_fallback: bool = True,
):
    train_df = metadata[metadata["split"] == "train"].copy()
    val_df = metadata[metadata["split"] == "val"].copy()
    test_df = metadata[metadata["split"] == "test"].copy()

    y_train_base = build_level_labels(train_df, level)
    y_val_base = build_level_labels(val_df, level)
    y_test_base = build_level_labels(test_df, level)

    y_train_raw, keep_train = apply_case(train_df, y_train_base, level, case_name)
    y_val_raw, keep_val = apply_case(val_df, y_val_base, level, case_name)
    y_test_raw, keep_test = apply_case(test_df, y_test_base, level, case_name)

    train_df = train_df.loc[keep_train].reset_index(drop=True)
    val_df = val_df.loc[keep_val].reset_index(drop=True)
    test_df = test_df.loc[keep_test].reset_index(drop=True)

    raw_edge_feature_names = RAW_EDGE_FEATURES[level]

    train_raw_mask, train_deep_raw_mask = _build_availability_masks(train_df, raw_edge_feature_names, deep_feature_subdir)
    val_raw_mask, val_deep_raw_mask = _build_availability_masks(val_df, raw_edge_feature_names, deep_feature_subdir)
    test_raw_mask, test_deep_raw_mask = _build_availability_masks(test_df, raw_edge_feature_names, deep_feature_subdir)

    deep_ready = train_deep_raw_mask.any() and val_deep_raw_mask.any() and test_deep_raw_mask.any()

    if deep_ready:
        node_feature_source = "deep"
        train_df, y_train_raw = _apply_mask(train_df, y_train_raw, train_deep_raw_mask)
        val_df, y_val_raw = _apply_mask(val_df, y_val_raw, val_deep_raw_mask)
        test_df, y_test_raw = _apply_mask(test_df, y_test_raw, test_deep_raw_mask)
    else:
        if not allow_raw_node_fallback:
            raise RuntimeError(
                "Deep feature files are insufficient across train/val/test after filtering. "
                "Enable allow_raw_node_fallback or generate missing deep features."
            )
        node_feature_source = "raw"
        train_df, y_train_raw = _apply_mask(train_df, y_train_raw, train_raw_mask)
        val_df, y_val_raw = _apply_mask(val_df, y_val_raw, val_raw_mask)
        test_df, y_test_raw = _apply_mask(test_df, y_test_raw, test_raw_mask)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise RuntimeError(
            f"No usable samples after feature-file filtering. "
            f"Counts train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}"
        )

    le = LabelEncoder()
    le.fit(np.concatenate([y_train_raw, y_val_raw, y_test_raw]))

    y_train = le.transform(y_train_raw)
    y_val = le.transform(y_val_raw)
    y_test = le.transform(y_test_raw)

    train_dataset = FGVDGraphDataset(
        train_df,
        y_train,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )
    val_dataset = FGVDGraphDataset(
        val_df,
        y_val,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )
    test_dataset = FGVDGraphDataset(
        test_df,
        y_test,
        raw_edge_feature_names=raw_edge_feature_names,
        deep_feature_subdir=deep_feature_subdir,
        node_feature_source=node_feature_source,
    )

    loader_kwargs = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": device.type == "cuda",
    }
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 4

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

    sample_source_df = train_df if len(train_df) > 0 else (val_df if len(val_df) > 0 else test_df)
    sample_vehicle_id = str(sample_source_df.iloc[0]["vehicle_id"])
    sample_split = str(sample_source_df.iloc[0]["split"])

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "label_encoder": le,
        "train_count": len(train_df),
        "val_count": len(val_df),
        "test_count": len(test_df),
        "num_classes": len(le.classes_),
        "sample_vehicle_id": sample_vehicle_id,
        "sample_split": sample_split,
        "deep_feature_subdir": deep_feature_subdir,
        "raw_edge_feature_names": raw_edge_feature_names,
        "node_feature_source": node_feature_source,
    }


# -------------------------
# Plotting
# -------------------------
def plot_learning_curves(history: dict, title: str, out_path: Path = None):
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(epochs, history["train_loss"], label="train_loss", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], label="val_loss", linewidth=2)
    axes[0].set_title("Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="train_acc", linewidth=2)
    axes[1].plot(epochs, history["val_acc"], label="val_acc", linewidth=2)
    axes[1].set_title("Accuracy Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()


# -------------------------
# Main fit function
# -------------------------
def fit_with_resume(
    level: str = "L1",
    case_name: str = "all",
    additional_epochs: int = 20,
    target_total_epochs: int = None,
    resume: bool = True,
    clear_old_checkpoints: bool = False,
    batch_size: int = 32,
    hidden_dim: int = 64,
    num_layers: int = 2,
    dropout: float = 0.5,
    lr: float = 1e-3,
    num_workers: int = 0,
    print_every: int = 5,
    deep_feature_subdir: str = DEEP_NODE_FEATURE_SUBDIR,
    edge_sigma: float = 0.5,
    allow_raw_node_fallback: bool = True,
    use_amp: bool = True,
    run_name: str = "fresh_start",
    save_curve: bool = False,
):
    run_tag = str(run_name).strip().replace(" ", "_") or "fresh_start"
    ckpt_root = Path("checkpoints/sgcn_df") / run_tag
    ckpt_root.mkdir(parents=True, exist_ok=True)

    curve_png = None
    if save_curve:
        plot_root = Path("plots/sgcn_df") / run_tag
        plot_root.mkdir(parents=True, exist_ok=True)

    tag = _tag(level, case_name)
    last_ckpt = ckpt_root / f"{tag}_last.pt"
    best_ckpt = ckpt_root / f"{tag}_best.pt"
    if save_curve:
        curve_png = plot_root / f"{tag}_curves.png"

    if clear_old_checkpoints:
        if last_ckpt.exists():
            last_ckpt.unlink()
        if best_ckpt.exists():
            best_ckpt.unlink()

    exp = prepare_experiment(
        level,
        case_name,
        batch_size=batch_size,
        num_workers=num_workers,
        deep_feature_subdir=deep_feature_subdir,
        allow_raw_node_fallback=allow_raw_node_fallback,
    )
    print(f"\nRunning {level} | {case_name}")
    print(f"Samples: train={exp['train_count']}, val={exp['val_count']}, test={exp['test_count']}")
    print(f"Classes: {exp['num_classes']}")
    print(f"Node features source: {exp['node_feature_source']}")
    print(f"Node deep subdir: {exp['deep_feature_subdir']}")
    print(f"Edge features (raw): {exp['raw_edge_feature_names']}")
    print(f"Gaussian sigma: {edge_sigma}")
    print(f"AMP enabled: {use_amp and device.type == 'cuda'}")

    sample_x = load_node_features(
        exp["sample_vehicle_id"],
        exp["sample_split"],
        exp["deep_feature_subdir"],
        exp["raw_edge_feature_names"],
        exp["node_feature_source"],
    )
    in_channels = int(sample_x.shape[1])

    model = SGCN(
        in_channels=in_channels,
        num_classes=exp["num_classes"],
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
        edge_sigma=edge_sigma,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_f1": [],
        "val_f1": [],
    }

    start_epoch = 1
    best_val_acc = -1.0
    best_epoch = 0
    best_model_state = None

    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        history = ckpt.get("history", history)
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val_acc = float(ckpt.get("best_val_acc", -1.0))
        best_epoch = int(ckpt.get("best_epoch", 0))
        best_model_state = ckpt.get("best_model_state_dict", None)

        saved_classes = ckpt.get("label_classes", [])
        current_classes = exp["label_encoder"].classes_.tolist()
        if saved_classes and saved_classes != current_classes:
            raise RuntimeError("Label encoding changed since checkpoint creation. Cannot safely resume.")

        print(f"Resumed from checkpoint: {last_ckpt}")
        print(f"Restarting at epoch {start_epoch}")

    if target_total_epochs is not None:
        end_epoch = int(target_total_epochs)
    else:
        end_epoch = start_epoch + int(additional_epochs) - 1

    if start_epoch > end_epoch:
        print("No new epochs requested. Evaluating current best model.")
    else:
        for epoch in range(start_epoch, end_epoch + 1):
            tr_loss, tr_acc, tr_f1, _, _ = run_epoch(model, exp["train_loader"], criterion, optimizer, use_amp=use_amp)
            va_loss, va_acc, va_f1, _, _ = run_epoch(model, exp["val_loader"], criterion, optimizer=None, use_amp=use_amp)

            history["epoch"].append(epoch)
            history["train_loss"].append(float(tr_loss))
            history["val_loss"].append(float(va_loss))
            history["train_acc"].append(float(tr_acc))
            history["val_acc"].append(float(va_acc))
            history["train_f1"].append(float(tr_f1))
            history["val_f1"].append(float(va_f1))

            if va_acc > best_val_acc:
                best_val_acc = float(va_acc)
                best_epoch = int(epoch)
                best_model_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

                torch.save(
                    {
                        "epoch": epoch,
                        "best_epoch": best_epoch,
                        "best_val_acc": best_val_acc,
                        "best_model_state_dict": best_model_state,
                        "history": history,
                        "label_classes": exp["label_encoder"].classes_.tolist(),
                    },
                    best_ckpt,
                )

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_model_state_dict": best_model_state,
                    "history": history,
                    "label_classes": exp["label_encoder"].classes_.tolist(),
                },
                last_ckpt,
            )

            if device.type == "cuda":
                # Reduces long-run memory fragmentation between epochs.
                torch.cuda.empty_cache()

            if epoch == start_epoch or epoch % print_every == 0 or epoch == end_epoch:
                print(
                    f"Epoch {epoch:03d} | "
                    f"train_acc={tr_acc:.4f} val_acc={va_acc:.4f} | "
                    f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f}"
                )

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    te_loss, te_acc, te_f1, te_pred, te_true = run_epoch(model, exp["test_loader"], criterion, optimizer=None, use_amp=use_amp)

    pred_labels = exp["label_encoder"].inverse_transform(te_pred.astype(int))
    true_labels = exp["label_encoder"].inverse_transform(te_true.astype(int))

    print("\nFinal test metrics (best validation-accuracy checkpoint):")
    print(f"test_acc={te_acc:.4f} | test_f1={te_f1:.4f} | test_loss={te_loss:.4f}")
    print(classification_report(true_labels, pred_labels, digits=4))

    title = f"SGCN Learning Curves | {level} | {case_name}"
    plot_learning_curves(history, title=title, out_path=curve_png)

    if history["train_acc"]:
        gap = history["train_acc"][-1] - history["val_acc"][-1]
        if history["train_acc"][-1] < 0.70 and history["val_acc"][-1] < 0.70:
            fit_note = "Possible underfitting (both train/val accuracy are low)."
        elif gap > 0.08:
            fit_note = "Possible overfitting (train-val accuracy gap is high)."
        else:
            fit_note = "No strong over/underfitting signal from final epoch gap."
        print(f"Fit check: {fit_note} gap={gap:+.4f}")

    print(f"Saved: {last_ckpt}")
    print(f"Saved: {best_ckpt}")
    print(f"Checkpoint folder: {ckpt_root.resolve()}")
    if save_curve and curve_png is not None:
        print(f"Saved: {curve_png}")
        print(f"Plot folder: {plot_root.resolve()}")
    print(
        f"Checkpoint files present -> last: {last_ckpt.exists()} | best: {best_ckpt.exists()}"
    )

    return {
        "level": level,
        "case": case_name,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "test_acc": float(te_acc),
        "test_f1": float(te_f1),
        "checkpoint_last": str(last_ckpt),
        "checkpoint_best": str(best_ckpt),
        "edge_sigma": float(edge_sigma),
        "node_feature_subdir": deep_feature_subdir,
        "node_feature_source": exp["node_feature_source"],
        "amp": bool(use_amp and device.type == "cuda"),
    }


# -------------------------
# USER CONTROLS
# -------------------------
LEVEL_TO_RUN = "L1"
CASE_TO_RUN = "all"            # all, tw_vs_all, thw_vs_all, tw_thw_vs_all, tw_vs_car
RUN_NAME = "L1_all"
RESUME = True                   # resume from the existing checkpoint folder
CLEAR_OLD_CHECKPOINTS = False    # keep existing checkpoints and continue from them
ADDITIONAL_EPOCHS = 50          # used when TARGET_TOTAL_EPOCHS is None
TARGET_TOTAL_EPOCHS = 50        # exact total epochs target
EDGE_SIGMA = 0.5                # Gaussian kernel sigma for dynamic edge weights
ALLOW_RAW_NODE_FALLBACK = True  # if deep files are insufficient, use raw as node attributes
NUM_WORKERS = 0                 # Set to 0 to prevent RAM exhaustion / system hangs
USE_AMP = True                  # mixed precision on CUDA for faster training
SAVE_CURVE = False              # False => only keep *_best.pt and *_last.pt

result = fit_with_resume(
    level=LEVEL_TO_RUN,
    case_name=CASE_TO_RUN,
    additional_epochs=ADDITIONAL_EPOCHS,
    target_total_epochs=TARGET_TOTAL_EPOCHS,
    resume=RESUME,
    clear_old_checkpoints=CLEAR_OLD_CHECKPOINTS,
    batch_size=32,
    hidden_dim=64,
    num_layers=2,
    dropout=0.5,
    lr=1e-3,
    num_workers=NUM_WORKERS,
    print_every=5,
    deep_feature_subdir=DEEP_NODE_FEATURE_SUBDIR,
    edge_sigma=EDGE_SIGMA,
    allow_raw_node_fallback=ALLOW_RAW_NODE_FALLBACK,
    use_amp=USE_AMP,
    run_name=RUN_NAME,
    save_curve=SAVE_CURVE,
)

print("\nRun result:")
print(result)

Using device: cuda


FileNotFoundError: Missing adjacency: FGVD_Graph_Handover/master_grid_adj.npz